In [28]:
print("Hello world!")

Hello world!


In [29]:
%pip install pandas
%pip install numpy
%pip install imbalanced-learn
%pip install seaborn
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from solutions.Solution_3.data_gen import extract_infos, packsec

ModuleNotFoundError: No module named 'solutions'

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
df = pd.read_csv('../data/PE_Features.csv')
df.head()

In [ ]:
df.describe()

In [ ]:
features = df.drop(columns=['FileCategory'])
label = df['FileCategory']

In [ ]:
sns.heatmap(features.corr().abs())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, label, test_size=0.3, random_state=42, stratify=label
)

In [ ]:
rfc = RandomForestClassifier(n_estimators=200, random_state=42)
rfc.fit(X_train, y_train)

In [ ]:
rfc.feature_importances_

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rfc.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance_df

In [ ]:
importance_df.shape

In [ ]:
top_k = 15

selected_features = importance_df["Feature"].head(top_k).tolist()

X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [ ]:
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', class_weight='balanced'))
])

svm_model.fit(X_train_selected, y_train)

In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000)),
])

lr_model.fit(X_train_selected, y_train)

In [ ]:
rf_model = Pipeline([
    ('scaler', StandardScaler()),
    ('rfc', RandomForestClassifier(n_estimators=300, random_state=42))
])

rf_model.fit(X_train_selected, y_train)

In [ ]:
models = {
    "Random Forest": rf_model,
    "SVM": svm_model,
    "Logistic Regression": lr_model
}

for name, model in models.items():
    y_pred = model.predict(X_test_selected)
    print("\n", name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))


In [ ]:
def predict_file(filepath):
    # Extract full feature list (same as during dataset generation)
    raw_features = extract_infos(filepath, packsec)

    # Full column list (must match dataset order exactly)
    columns = [
        "NumberOfSections",
        "NumberOfSymbols",
        "TimeDateStamp",
        "Characteristics",
        "AddressOfEntryPoint",
        "SizeOfInitializedData",
        "SizeOfUnInitializedData",
        "CheckSum",
        "DllCharacteristics",
        "PackedFlag",
        "NumberOfImportedDlls",
        "NumberOfImportedFunctions",
        "NumStrings",
        "AvgStringLength",
        "MaxStringLength",
        "NumURLs",
        "NumDLL",
        "NumEXE",
        "NumRegistry",
        "NumSuspiciousAPI"
    ]

    import pandas as pd

    df = pd.DataFrame([raw_features], columns=columns)

    # Select same top features used during training
    df_selected = df[selected_features]

    prediction = rf_model.predict(df_selected)[0]

    if prediction == 1:
        print("MALWARE DETECTED")
    else:
        print("File is BENIGN")

    return prediction
